In [ ]:
import sys
import os
from pathlib import Path
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from skimage.measure import label, regionprops
from importlib import reload

sys.path.append(os.path.abspath("../../"))

import Segmentation.Util.env_utils
reload(Segmentation.Util.env_utils)


Segmentation.Util.env_utils.load_segmentation_env()

RAW_DATA_DIR = os.getenv("RAW_DATA_DIR")
HIGHRES_IMG_DIR_NAME = os.getenv("HIGHRES_IMG_DIR_NAME", "images_4096")
HIGHRES_MASK_DIR_NAME = os.getenv("HIGHRES_MASK_DIR_NAME", "masks_4096")

mask_dir = Path(RAW_DATA_DIR) / HIGHRES_MASK_DIR_NAME

image_paths = list(mask_dir.glob("*.png"))

override_existing = False
min_size = 1
max_size = 300

for path in image_paths:
    img = Image.open(path).convert('L')
    img_np = np.array(img)

    binary = (img_np > 127).astype(np.uint8)
    labeled = label(binary)
    regions = regionprops(labeled)

    small_region_labels = [
        region.label for region in regions
        if min_size <= region.area < max_size
    ]

    if not small_region_labels:
        continue

    cleaned_binary = binary.copy()
    for lbl in small_region_labels:
        cleaned_binary[labeled == lbl] = 0

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(binary, cmap='gray')

    for region in regions:
        if region.label in small_region_labels:
            y, x = region.centroid
            radius = 40
            circ = Circle((x, y), radius=radius, edgecolor='red',
                          facecolor='none', linewidth=1.5)
            ax.add_patch(circ)

    ax.set_title(f"Removed small components in: {path.name}")
    ax.axis('off')
    plt.show()

    if override_existing:
        cleaned_img = Image.fromarray((cleaned_binary * 255).astype(np.uint8))
        cleaned_img.save(path)

loaded environment variables from: /local/scratch/jhehli/ForkSight/Segmentation/.env
